# Task 1 — Populate the Skill class from ESCO v1.2.1

This notebook reads the **English ESCO v1.2.1 CSV distribution**, keeps each skill's original IRI, and adds instances to the existing ontology's `Skill` class. It starts from the original schema on every run and saves a separate RDF/XML graph.

Only the following triples are added:

| CSV field | RDF mapping |
|---|---|
| `conceptUri` | Subject IRI; `rdf:type` → local `Skill` class |
| `preferredLabel`, `altLabels` | `rdfs:label` with `@en` |
| `description`, `definition` (when present) | `rdfs:comment` with `@en` |

Alternative labels are newline-separated in the CSV. Hidden labels, scope notes, relationships, skill subtypes, and other metadata are deliberately excluded. Both knowledge and skill/competence concepts belong to ESCO's skill pillar and are included.

**IRI clarification:** the assignment PDF prints `model#Skil`; the [official ESCO model](https://webgate.acceptance.ec.europa.eu/esco/lod/static/model.html) calls the class `http://data.europa.eu/esco/model#Skill`. The skills CSV represents these concepts using `conceptType=KnowledgeSkillCompetence`. We map them to the local `Skill` class without importing the ESCO schema.

## 1. Setup and source data

1. From the repository root, run `uv sync`. If needed, copy `.env.example` to `.env`.
2. Select this project's `.venv/bin/python` as the notebook kernel.
3. Open the [official ESCO download page](https://esco.ec.europa.eu/en/use-esco/download). Select **ESCO dataset – v1.2.1**, **classification**, **CSV**, **English**, and follow the portal's download steps.
4. Extract `skills_en.csv` to `data/raw/esco/v1.2.1/skills_en.csv`, or set `ESCO_SKILLS_PATH` in `.env` to its location.
5. Run all cells in order.

The CSV does not contain a release identifier or language tags. Its version and language must therefore be selected correctly at download time; the directory name alone cannot prove the release. The notebook records the input SHA-256 for reproducibility. See also the [official dataset structure](https://esco.ec.europa.eu/en/structure-esco-downloadable-datasets).

No API key is required. Keep the downloaded CSV unchanged. The original ontology is read from `ONTOLOGY_PATH`; the result is written to `OUTPUT_ONTOLOGY_PATH`.

In [ ]:
import csv
import hashlib
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from rdflib import Graph, Literal, RDF, RDFS, OWL, URIRef
from rdflib.compare import isomorphic

env_file = find_dotenv(usecwd=True)
if not env_file:
    raise FileNotFoundError("Copy .env.example to .env in the repository root first.")
project_root = Path(env_file).resolve().parent
load_dotenv(env_file)


def project_path(value: str) -> Path:
    return (project_root / Path(value).expanduser()).resolve()


data_dir = project_path(os.environ["DATA_DIR"])
ontology_path = project_path(os.environ["ONTOLOGY_PATH"])
output_ontology_path = project_path(os.environ["OUTPUT_ONTOLOGY_PATH"])
esco_skills_path = project_path(os.environ.get(
    "ESCO_SKILLS_PATH", str(data_dir / "raw" / "esco" / "v1.2.1" / "skills_en.csv")
))

if output_ontology_path in {ontology_path, esco_skills_path}:
    raise ValueError("The output path must differ from both input paths.")
for input_path in (ontology_path, esco_skills_path):
    if not input_path.is_file():
        raise FileNotFoundError(
            f"Missing input: {input_path}. See the setup instructions above."
        )

schema_sha256 = hashlib.sha256(ontology_path.read_bytes()).hexdigest()
print(f"ESCO release: v1.2.1 (English; selected at download)")
print(f"CSV: {esco_skills_path}")
print(f"CSV SHA-256: {hashlib.sha256(esco_skills_path.read_bytes()).hexdigest()}")
print(f"Output: {output_ontology_path}")

## 2. Load the existing schema

Discover the namespace from the declared OWL class named `Skill` instead of hard-coding a machine-specific or guessed ontology namespace. The original schema triples are preserved; no ontology imports are followed.

In [ ]:
schema = Graph().parse(ontology_path, format="xml")
skill_classes = [
    subject for subject in schema.subjects(RDF.type, OWL.Class)
    if isinstance(subject, URIRef)
    and str(subject).rsplit("#", 1)[-1].rsplit("/", 1)[-1] == "Skill"
]
if len(skill_classes) != 1:
    raise ValueError(f"Expected one OWL class named Skill; found {skill_classes}.")
SKILL_CLASS = skill_classes[0]
print(f"Local Skill class: {SKILL_CLASS}")
print(f"Original schema: {len(schema):,} triples")

## 3. Read and check the English skills CSV

Validate the headers, concept types, original IRIs, and required labels before generating any output. Read with `utf-8-sig` to accept a UTF-8 BOM. Do not filter on `skillType`: that would incorrectly drop knowledge concepts. Definitions may be empty; empty literals are never added.

In [ ]:
def read_skills(path: Path) -> list[dict[str, str]]:
    required = {
        "conceptType", "conceptUri", "preferredLabel", "altLabels",
        "description", "definition",
    }
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Missing CSV columns: {sorted(missing)}")
        rows = []
        seen = set()
        for row_number, row in enumerate(reader, start=2):
            if None in row or any(row.get(key) is None for key in required):
                raise ValueError(f"Malformed CSV record {row_number}.")
            if row["conceptType"].strip() != "KnowledgeSkillCompetence":
                raise ValueError(f"Unexpected concept type in record {row_number}.")
            iri = row["conceptUri"]
            if (not iri.startswith("http://data.europa.eu/esco/skill/")
                    or not iri.removeprefix("http://data.europa.eu/esco/skill/")
                    or any(c.isspace() for c in iri)):
                raise ValueError(f"Invalid ESCO skill IRI in record {row_number}: {iri!r}")
            if iri in seen:
                raise ValueError(f"Duplicate skill IRI in record {row_number}: {iri}")
            if not row["preferredLabel"].strip():
                raise ValueError(f"Missing preferred label in record {row_number}.")
            seen.add(iri)
            rows.append(row)
    if not rows:
        raise ValueError("The skills CSV is empty.")
    return rows


skills = read_skills(esco_skills_path)
print(f"Loaded {len(skills):,} distinct ESCO skill concepts.")

## 4. Build the population triples

Keep the population in a separate graph so it can be checked before merging with the schema. RDF graphs automatically deduplicate identical triples, including repeated alternative labels or identical descriptions and definitions. Description and definition paragraphs are kept intact.

In [ ]:
def populate_skills(rows: list[dict[str, str]], skill_class: URIRef) -> Graph:
    population = Graph()
    for row in rows:
        skill = URIRef(row["conceptUri"])
        population.add((skill, RDF.type, skill_class))
        labels = [row["preferredLabel"], *row["altLabels"].splitlines()]
        for text in labels:
            if text.strip():
                population.add((skill, RDFS.label, Literal(text.strip(), lang="en")))
        for field in ("description", "definition"):
            text = row[field].strip()
            if text:
                population.add((skill, RDFS.comment, Literal(text, lang="en")))
    return population


population = populate_skills(skills, SKILL_CLASS)
print(f"Population triples: {len(population):,}")

## 5. Validate coverage and the “nothing else” restriction

Check every skill against its source record, including all alternative labels, descriptions, and optional definitions. Also reject any extra subjects, predicates, types, or non-English annotation literals.

In [ ]:
source_iris = {URIRef(row["conceptUri"]) for row in skills}
assert set(population.subjects(RDF.type, SKILL_CLASS)) == source_iris
assert set(population.subjects()) == source_iris
assert set(population.predicates()) <= {RDF.type, RDFS.label, RDFS.comment}

for subject, predicate, value in population:
    if predicate == RDF.type:
        assert value == SKILL_CLASS
    else:
        assert isinstance(value, Literal) and value.language == "en"
        assert str(value).strip()

for row in skills:
    subject = URIRef(row["conceptUri"])
    expected_labels = {
        Literal(text.strip(), lang="en")
        for text in [row["preferredLabel"], *row["altLabels"].splitlines()]
        if text.strip()
    }
    expected_comments = {
        Literal(row[field].strip(), lang="en")
        for field in ("description", "definition") if row[field].strip()
    }
    assert set(population.objects(subject, RDFS.label)) == expected_labels
    assert set(population.objects(subject, RDFS.comment)) == expected_comments

print(f"Validated all {len(source_iris):,} skills: full coverage and only permitted triples.")
print(f"Labels: {sum(1 for _ in population.triples((None, RDFS.label, None))):,}")
print(f"Comments: {sum(1 for _ in population.triples((None, RDFS.comment, None))):,}")
print(f"Skills with definitions: {sum(bool(row['definition'].strip()) for row in skills):,}")
print(f"Skills without descriptions: {sum(not row['description'].strip() for row in skills):,}")

## 6. Save and verify the populated ontology

Merge into a fresh copy of the original schema. Serialize to a temporary file, check the RDF/XML round trip, then replace the output. This avoids leaving a partial final file if serialization or validation fails. Re-running starts from the original schema rather than the previous output.

In [ ]:
populated_graph = Graph()
for prefix, namespace in schema.namespaces():
    populated_graph.bind(prefix, namespace)
populated_graph += schema
populated_graph += population

assert all(triple in populated_graph for triple in schema)
assert set(populated_graph) == set(schema) | set(population)

output_ontology_path.parent.mkdir(parents=True, exist_ok=True)
# A unique temporary file in the output directory also permits atomic replacement.
from tempfile import NamedTemporaryFile

with NamedTemporaryFile(
    dir=output_ontology_path.parent, suffix=".rdf", delete=False
) as handle:
    temporary_path = Path(handle.name)
try:
    populated_graph.serialize(destination=temporary_path, format="xml", encoding="utf-8")
    reloaded = Graph().parse(temporary_path, format="xml")
    assert isomorphic(populated_graph, reloaded), "RDF/XML round-trip changed the graph."
    assert hashlib.sha256(ontology_path.read_bytes()).hexdigest() == schema_sha256
    temporary_path.replace(output_ontology_path)
finally:
    temporary_path.unlink(missing_ok=True)

print(f"Saved {len(populated_graph):,} triples to {output_ontology_path}")
print("Original schema unchanged; saved RDF/XML verified.")

## 7. Inspect a sample

Each sample retains its ESCO IRI and has the local class membership, merged English labels, and available comments. Population statistics and sample output are displayed when the real dataset is run; no example data is substituted for ESCO.

In [ ]:
for subject in sorted(source_iris, key=str)[:3]:
    print(f"\n{subject}")
    print(f"  rdf:type: {SKILL_CLASS}")
    for label in sorted(population.objects(subject, RDFS.label), key=str):
        print(f"  label: {label}")
    for comment in sorted(population.objects(subject, RDFS.comment), key=str):
        print(f"  comment: {comment}")